## 1. Instalasi Dependency


In [25]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn

## 2. Setup Konfigurasi dan Library


In [26]:
import os, re, ast, json, random, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED             = 42
CPU_COUNT        = os.cpu_count() or 2
WORKERS          = max(1, CPU_COUNT - 1)
GPU_COUNT        = torch.cuda.device_count()
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_DIR        = Path("/kaggle/input")
OUTPUT_DIR       = Path("/kaggle/working/outputs_scenario1_bertopic_specter2")
EMBEDDING_LABEL  = "SPECTER2"
EMBEDDING_MODEL  = "allenai/specter2_base"
MCS_CANDIDATES   = [20, 30, 40, 50, 60, 70, 80, 100, 120]

random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[env] = str(CPU_COUNT)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print(f"CPU={CPU_COUNT} | WORKERS={WORKERS} | GPU={GPU_COUNT}x ({DEVICE}) | Embedding={EMBEDDING_LABEL}")
print(f"MCS candidates: {MCS_CANDIDATES}")

CPU=4 | WORKERS=3 | GPU=2x (cuda) | Embedding=SPECTER2
MCS candidates: [20, 30, 40, 50, 60, 70, 80, 100, 120]


## 3. Load Dataset


In [27]:
def find_dataset(base):
    matches = list(base.rglob("dataset_pubmed_clean.parquet"))
    if matches: return matches[0]
    raise FileNotFoundError("dataset_pubmed_clean.parquet not found in /kaggle/input")

df = pd.read_parquet(find_dataset(INPUT_DIR))
print(f"Shape: {df.shape}")

Shape: (26046, 9)


## 4. Prepare Text & Labels


In [28]:
def get_col(df, names, required=False):
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m: return m[n.lower()]
    if required: raise ValueError(f"Column not found: {names}")
    return None

def parse_labels(v):
    if isinstance(v, (list, np.ndarray)): return list(v)
    try:
        if pd.isna(v): return []
    except (TypeError, ValueError): pass
    s = str(v).strip()
    try:
        p = ast.literal_eval(s)
        if isinstance(p, list): return [str(x) for x in p]
    except: pass
    return [x.strip() for x in re.split(r"[;,|]", s) if x.strip()]

def light_clean(text):
    t = str(text).lower()
    t = re.sub(r"<.*?>", " ", t)
    return re.sub(r"\s+", " ", t).strip()

col_pmid = get_col(df, ["pmid", "paper_id", "id"])
col_abs  = get_col(df, ["abstract", "clean_abstract"], required=True)
col_lbl  = get_col(df, ["final_labels"])

data = pd.DataFrame({
    "paper_id"   : df[col_pmid].astype(str) if col_pmid else np.arange(len(df)).astype(str),
    "abstract"   : df[col_abs].fillna("").astype(str),
    "label_list" : df[col_lbl].apply(parse_labels) if col_lbl else [[] for _ in range(len(df))],
})
data["primary_label"] = data["label_list"].apply(lambda x: x[0] if x else "Unknown")
data = data[data["primary_label"] != "Unknown"].reset_index(drop=True)

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    data["doc"] = list(ex.map(light_clean, data["abstract"], chunksize=500))

data = (data[data["doc"].str.split().str.len() >= 5]
           .drop_duplicates("doc")
           .reset_index(drop=True))
docs   = data["doc"].tolist()
y_true = data["primary_label"].tolist()

print(f"Rows={len(data)} | Labels={data['primary_label'].nunique()}")

Rows=26018 | Labels=10


## 5. Build Embeddings (GPU T4 x2)


In [29]:
from sentence_transformers import SentenceTransformer

emb_path = OUTPUT_DIR / f"embeddings_{EMBEDDING_LABEL.lower()}_scenario1.npy"

if emb_path.exists():
    print("Loading cached embeddings...")
    embeddings = np.load(emb_path)
else:
    model_emb = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
    model_emb.max_seq_length = 512
    if GPU_COUNT > 1:
        print(f"Encoding with {GPU_COUNT} GPUs...")
        pool = model_emb.start_multi_process_pool(
            target_devices=[f"cuda:{i}" for i in range(GPU_COUNT)])
        embeddings = model_emb.encode_multi_process(
            docs, pool, batch_size=128, show_progress_bar=True)
        model_emb.stop_multi_process_pool(pool)
    else:
        print(f"Encoding with single {DEVICE}...")
        embeddings = model_emb.encode(
            docs, batch_size=128, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True)
    embeddings = np.asarray(embeddings, dtype=np.float32)
    np.save(emb_path, embeddings)

print(f"Embeddings: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding with 2 GPUs...


Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

Embeddings: (26018, 768)


## 6. Evaluation Setup


In [30]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score
from scipy.optimize import linear_sum_assignment
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

BIOMED_STOP = {
    "patient","patients","study","studies","result","results",
    "method","methods","conclusion","conclusions","background",
    "objective","objectives","data","analysis","significant",
    "significantly","associated","compared","comparison",
    "included","including","use","used","using","group","groups",
    "showed","shown"
}
STOP_ALL = set(STOPWORDS) | BIOMED_STOP

def tokenize(text):
    return [t for t in simple_preprocess(text, deacc=True, min_len=3) if t not in STOP_ALL]

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    tokenized_docs = list(ex.map(tokenize, docs, chunksize=500))

gensim_dict = Dictionary(tokenized_docs)
gensim_dict.filter_extremes(no_below=5, no_above=0.5, keep_n=50_000)

def topic_uniqueness(tw):
    all_w = [w for t in tw for w in t]
    return round(len(set(all_w)) / len(all_w), 4) if all_w else 0.0

def hungarian_f1(y_true, y_topic):
    labels = sorted(set(y_true))
    topics = sorted(set(y_topic))
    l2i = {l: i for i, l in enumerate(labels)}
    t2i = {t: i for i, t in enumerate(topics)}
    yt  = np.array([l2i[y] for y in y_true])
    yp  = np.array([t2i[y] for y in y_topic])
    M   = np.zeros((len(topics), len(labels)))
    for t in topics:
        for l in labels:
            M[t2i[t], l2i[l]] = f1_score((yt == l2i[l]).astype(int),
                                          (yp == t2i[t]).astype(int), zero_division=0)
    ri, ci = linear_sum_assignment(-M)
    t2l = {topics[r]: labels[c] for r, c in zip(ri, ci)}
    fb  = pd.Series(y_true).mode()[0]
    return round(f1_score(y_true, [t2l.get(t, fb) for t in y_topic], average="macro", zero_division=0), 4)

def extract_topic_words(tm, topn=10):
    return [
        [w for w, _ in tm.get_topic(t)[:topn] if isinstance(w, str) and len(w) > 1]
        for t in sorted(x for x in tm.get_topics() if x != -1)
    ]

def coherence_cv(tw):
    if not tw: return 0.0
    return round(CoherenceModel(topics=tw, texts=tokenized_docs, dictionary=gensim_dict,
                                coherence="c_v", processes=WORKERS).get_coherence(), 4)

print("Evaluation setup ready.")

Evaluation setup ready.


## 7. UMAP Sekali — Dipakai Semua Iterasi Sweep


In [31]:
from umap import UMAP

print("Running UMAP (sekali untuk semua sweep)...")
reduced = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric="cosine", random_state=SEED, n_jobs=WORKERS
).fit_transform(embeddings)

print(f"Reduced: {reduced.shape}")

Running UMAP (sekali untuk semua sweep)...
Reduced: (26018, 5)


## 8. Sweep MIN_CLUSTER_SIZE


In [32]:
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from hdbscan import HDBSCAN

vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | BIOMED_STOP), ngram_range=(1, 2), min_df=1
)

sweep_results = []

for mcs in MCS_CANDIDATES:
    print(f"MCS={mcs} ...", end=" ", flush=True)
    hdbscan_model = HDBSCAN(
        min_cluster_size=mcs, metric="euclidean",
        cluster_selection_method="eom", prediction_data=True,
        core_dist_n_jobs=WORKERS
    )
    tm = BERTopic(
        umap_model=BaseDimensionalityReduction(),
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=15, calculate_probabilities=False, verbose=False
    )
    topics, _ = tm.fit_transform(docs, embeddings=reduced)
    topics = list(topics)

    k    = len(set(topics)) - (1 if -1 in topics else 0)
    out  = round(float(np.mean(np.array(topics) == -1)), 4)
    tw   = extract_topic_words(tm, topn=10)
    cv   = coherence_cv(tw)
    f1   = hungarian_f1(y_true, topics)
    uniq = topic_uniqueness(tw)
    hm   = round(2 * cv * f1 / (cv + f1), 4) if (cv + f1) > 0 else 0.0

    sweep_results.append({
        "min_cluster_size": mcs, "n_topics": k, "outlier_rate": out,
        "cv": cv, "topic_uniqueness": uniq, "hungarian_f1": f1, "harmonic_mean_cv_f1": hm
    })
    print(f"K={k} | Outlier={out} | Cv={cv} | F1={f1} | HM={hm}")

sweep_df = pd.DataFrame(sweep_results)
print("\n=== Sweep Results ===")
display(sweep_df.sort_values("harmonic_mean_cv_f1", ascending=False))

MCS=20 ... K=100 | Outlier=0.3872 | Cv=0.7299 | F1=0.3367 | HM=0.4608
MCS=30 ... K=54 | Outlier=0.2654 | Cv=0.7316 | F1=0.4499 | HM=0.5572
MCS=40 ... K=42 | Outlier=0.2422 | Cv=0.7345 | F1=0.4857 | HM=0.5847
MCS=50 ... K=36 | Outlier=0.2454 | Cv=0.7398 | F1=0.4909 | HM=0.5902
MCS=60 ... K=28 | Outlier=0.2095 | Cv=0.7432 | F1=0.4617 | HM=0.5696
MCS=70 ... K=26 | Outlier=0.1971 | Cv=0.7515 | F1=0.4633 | HM=0.5732
MCS=80 ... K=19 | Outlier=0.2016 | Cv=0.7026 | F1=0.4957 | HM=0.5813
MCS=100 ... K=18 | Outlier=0.1854 | Cv=0.7016 | F1=0.4962 | HM=0.5813
MCS=120 ... K=17 | Outlier=0.2037 | Cv=0.7116 | F1=0.492 | HM=0.5818

=== Sweep Results ===


,min_cluster_size,n_topics,outlier_rate,cv,topic_uniqueness,hungarian_f1,harmonic_mean_cv_f1
3,50,36,0.2454,0.7398,0.7444,0.4909,0.5902
2,40,42,0.2422,0.7345,0.7381,0.4857,0.5847
8,120,17,0.2037,0.7116,0.7824,0.4920,0.5818
7,100,18,0.1854,0.7016,0.7667,0.4962,0.5813
6,80,19,0.2016,0.7026,0.7632,0.4957,0.5813
5,70,26,0.1971,0.7515,0.7846,0.4633,0.5732
4,60,28,0.2095,0.7432,0.7679,0.4617,0.5696
1,30,54,0.2654,0.7316,0.7222,0.4499,0.5572
0,20,100,0.3872,0.7299,0.6860,0.3367,0.4608


## 9. Pilih Best MCS → Training Final


In [33]:
best_row = sweep_df.sort_values("harmonic_mean_cv_f1", ascending=False).iloc[0]
BEST_MCS = int(best_row["min_cluster_size"])

print(f"BEST MCS={BEST_MCS} | K={int(best_row['n_topics'])} | "
      f"Cv={best_row['cv']} | F1={best_row['hungarian_f1']} | HM={best_row['harmonic_mean_cv_f1']}")

# Training final dengan MCS terbaik
hdbscan_final = HDBSCAN(
    min_cluster_size=BEST_MCS, metric="euclidean",
    cluster_selection_method="eom", prediction_data=True,
    core_dist_n_jobs=WORKERS
)
topic_model = BERTopic(
    umap_model=BaseDimensionalityReduction(),
    hdbscan_model=hdbscan_final,
    vectorizer_model=vectorizer_model,
    top_n_words=15, calculate_probabilities=True, verbose=True
)
topics, probs = topic_model.fit_transform(docs, embeddings=reduced)
topics = list(topics)

natural_k    = len(set(topics)) - (1 if -1 in topics else 0)
outlier_rate = round(float(np.mean(np.array(topics) == -1)), 4)
topic_words  = extract_topic_words(topic_model, topn=10)
cv           = coherence_cv(topic_words)
f1           = hungarian_f1(y_true, topics)

results_df = pd.DataFrame([{
    "experiment"      : f"BERTopic_{EMBEDDING_LABEL}",
    "embedding_model" : EMBEDDING_LABEL,
    "model"           : "BERTopic_HDBSCAN_Natural",
    "K"               : natural_k,
    "k_type"          : "natural",
    "n_topics_found"  : natural_k,
    "outlier_rate"    : outlier_rate,
    "cv"              : cv,
    "topic_uniqueness": topic_uniqueness(topic_words),
    "hungarian_f1"    : f1,
    "harmonic_mean_cv_f1": round(2*cv*f1/(cv+f1), 4) if (cv+f1) > 0 else 0.0,
    "best_min_cluster_size": BEST_MCS,
}])

print(f"\nFinal: K={natural_k} | Outlier={outlier_rate} | Cv={cv} | F1={f1}")
display(results_df)

2026-06-13 19:04:02,980 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-13 19:04:02,980 - BERTopic - Dimensionality - Completed ✓
2026-06-13 19:04:02,982 - BERTopic - Cluster - Start clustering the reduced embeddings


BEST MCS=50 | K=36 | Cv=0.7398 | F1=0.4909 | HM=0.5902


2026-06-13 19:04:06,876 - BERTopic - Cluster - Completed ✓
2026-06-13 19:04:06,885 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-13 19:04:28,539 - BERTopic - Representation - Completed ✓



Final: K=36 | Outlier=0.2454 | Cv=0.7398 | F1=0.4909


,experiment,embedding_model,model,K,k_type,n_topics_found,outlier_rate,cv,topic_uniqueness,hungarian_f1,harmonic_mean_cv_f1,best_min_cluster_size
0,BERTopic_SPECTER2,SPECTER2,BERTopic_HDBSCAN_Natural,36,natural,36,0.2454,0.7398,0.7444,0.4909,0.5902,50


## 10. Save Output


In [35]:
topic_rows = [
    {
        "experiment": f"BERTopic_{EMBEDDING_LABEL}", "embedding_model": EMBEDDING_LABEL,
        "model": "BERTopic_HDBSCAN_Natural", "K": int(natural_k), "k_type": "natural",
        "topic_id": tid,
        "top_words": ", ".join(w for w, s in words[:15] if isinstance(w, str)),
        "weights": json.dumps({w: round(float(s), 6) for w, s in words[:15] if isinstance(w, str)})
    }
    for tid, words in topic_model.get_topics().items() if tid != -1
]

lbl = EMBEDDING_LABEL.lower()
results_df.to_csv(OUTPUT_DIR / f"bertopic_{lbl}_scenario1_results.csv",      index=False)
pd.DataFrame(topic_rows).to_csv(OUTPUT_DIR / f"bertopic_{lbl}_scenario1_topics.csv",       index=False)
data.to_csv(OUTPUT_DIR / f"bertopic_{lbl}_scenario1_dataset_used.csv",  index=False)
sweep_df.to_csv(OUTPUT_DIR / f"bertopic_{lbl}_mcs_sweep_results.csv",   index=False)

print(f"Saved to {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

Saved to /kaggle/working/outputs_scenario1_bertopic_specter2
  bertopic_specter2_mcs_sweep_results.csv
  bertopic_specter2_scenario1_dataset_used.csv
  bertopic_specter2_scenario1_results.csv
  bertopic_specter2_scenario1_topics.csv
  embeddings_specter2_scenario1.npy
